# 11_mlflow_package_register_v2

Production MLflow packaging and Unity Catalog registration for the World Bank GEP Intelligence Agent.

This V2 notebook fixes the Model Serving import-path failure by building and validating a real `serving_runtime` Python package before MLflow logging.


In [0]:
# ============================================================
# CELL 1 — Install/confirm production dependencies
# ============================================================
%pip install -q "mlflow>=3.12.0" "databricks-sdk>=0.102.0" databricks-openai databricks-ai-search pandas pydantic


If the `%pip` cell updates packages in your session, restart Python once before continuing.


In [0]:
# ============================================================
# CELL 2 — Imports and configuration
# ============================================================
from pathlib import Path
import shutil
import sys
import importlib
import re

import mlflow
import pandas as pd
from mlflow.models import infer_signature

MODEL_NAME = "worldbank_ai.ai.gep_intelligence_agent"

SOURCE_DIR = Path(
    "/Workspace/Users/darsinilakshmiah@gmail.com/"
    "global-economic-prospects-agent/serving_runtime"
)

# Build a real Python package outside the Workspace notebook namespace.
PACKAGE_ROOT = Path("/tmp/worldbank_model_package_v2")
PACKAGE_DIR = PACKAGE_ROOT / "serving_runtime"

RUNTIME_FILES = [
    "supervisor_runtime.py",
    "data_agent_sql_runtime.py",
    "research_agent_runtime.py",
    "synthesis_agent_runtime.py",
    "guardrails_runtime.py",
    "orchestrator.py",
    "worldbank_model.py",
]

print("Model:", MODEL_NAME)
print("Source:", SOURCE_DIR)
print("Package root:", PACKAGE_ROOT)


In [0]:
# ============================================================
# CELL 3 — Validate source files and build real package
# ============================================================
missing = [name for name in RUNTIME_FILES if not (SOURCE_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing runtime files: {missing}")

if PACKAGE_ROOT.exists():
    shutil.rmtree(PACKAGE_ROOT)
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

for filename in RUNTIME_FILES:
    src = SOURCE_DIR / filename
    dst = PACKAGE_DIR / filename
    dst.write_text(src.read_text(encoding="utf-8"), encoding="utf-8")
    print(f"COPIED: {filename}")

(PACKAGE_DIR / "__init__.py").write_text(
    "# World Bank GEP production serving runtime package.\n",
    encoding="utf-8",
)
print("CREATED: __init__.py")


In [0]:
# ============================================================
# CELL 4 — Normalize local imports in the packaged COPY only
# ============================================================
# We do not change agent logic. We only make local imports package-safe.

orch_path = PACKAGE_DIR / "orchestrator.py"
orch = orch_path.read_text(encoding="utf-8")
replacements = {
    "from supervisor_runtime import": "from serving_runtime.supervisor_runtime import",
    "from data_agent_sql_runtime import": "from serving_runtime.data_agent_sql_runtime import",
    "from research_agent_runtime import": "from serving_runtime.research_agent_runtime import",
    "from synthesis_agent_runtime import": "from serving_runtime.synthesis_agent_runtime import",
    "from guardrails_runtime import": "from serving_runtime.guardrails_runtime import",
}
for old, new in replacements.items():
    orch = orch.replace(old, new)
orch_path.write_text(orch, encoding="utf-8")

model_path = PACKAGE_DIR / "worldbank_model.py"
model_text = model_path.read_text(encoding="utf-8")
model_text = model_text.replace(
    "from orchestrator import execute_agent",
    "from serving_runtime.orchestrator import execute_agent",
)
model_path.write_text(model_text, encoding="utf-8")

print("Normalized package imports in orchestrator.py and worldbank_model.py")


In [0]:
# ============================================================
# CELL 5 — Validate there are no unresolved top-level local imports
# ============================================================
local_modules = {Path(name).stem for name in RUNTIME_FILES}
problems = []

for path in sorted(PACKAGE_DIR.glob("*.py")):
    for line_no, line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        stripped = line.strip()
        for module in local_modules:
            if re.match(rf"^from\s+{re.escape(module)}\s+import", stripped):
                problems.append((path.name, line_no, stripped))
            if re.match(rf"^import\s+{re.escape(module)}(?:\s|$)", stripped):
                problems.append((path.name, line_no, stripped))

if problems:
    for problem in problems:
        print("UNRESOLVED:", problem)
    raise RuntimeError("Unresolved top-level local imports remain.")

print("PASS: no unresolved top-level local imports")


In [0]:
# ============================================================
# CELL 6 — Import-test the real package
# ============================================================
package_root_str = str(PACKAGE_ROOT)
if package_root_str not in sys.path:
    sys.path.insert(0, package_root_str)

for name in list(sys.modules):
    if name == "serving_runtime" or name.startswith("serving_runtime."):
        del sys.modules[name]
importlib.invalidate_caches()

import serving_runtime
print("Package loaded from:", serving_runtime.__file__)

if str(PACKAGE_DIR) not in str(serving_runtime.__file__):
    raise RuntimeError("serving_runtime loaded from the wrong location")

from serving_runtime.supervisor_runtime import run_supervisor_agent
print("PASS: supervisor_runtime")
from serving_runtime.data_agent_sql_runtime import run_data_agent
print("PASS: data_agent_sql_runtime")
from serving_runtime.research_agent_runtime import run_research_agent
print("PASS: research_agent_runtime")
from serving_runtime.synthesis_agent_runtime import run_synthesis_agent
print("PASS: synthesis_agent_runtime")
from serving_runtime.guardrails_runtime import apply_pre_execution_guardrails, apply_post_execution_guardrails
print("PASS: guardrails_runtime")
from serving_runtime.orchestrator import execute_agent
print("PASS: orchestrator")

print("\n==========================================")
print("REAL PYTHON PACKAGE TEST PASSED")
print("==========================================")


In [0]:
# ============================================================
# CELL 7 — Define stable serving signature
# ============================================================
input_example = pd.DataFrame([
    {
        "question": "Show India's GDP growth from 2015 to 2025.",
        "conversation_context": "",
    }
])

# Schema-only example; this is not presented as a real prediction.
output_example = pd.DataFrame([
    {
        "status": "success",
        "route": "structured",
        "answer": "example",
        "total_latency_ms": 0.0,
        "plan_json": "{}",
        "citation_validation_json": "{}",
        "structured_json": "{}",
        "research_json": "{}",
    }
])

signature = infer_signature(input_example, output_example)
print(signature)


In [0]:
# ============================================================
# CELL 8 — Log V2 candidate with MLflow Models from Code
# ============================================================
mlflow.set_registry_uri("databricks-uc")

MODEL_FILE = str(PACKAGE_DIR / "worldbank_model.py")
PIP_REQUIREMENTS = [
    "mlflow>=3.12.0",
    "pandas",
    "pydantic",
    "databricks-sdk>=0.102.0",
    "databricks-openai",
    "databricks-ai-search",
]

with mlflow.start_run(run_name="worldbank-gep-production-v2") as run:
    model_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model=MODEL_FILE,
        # Package the directory ABOVE serving_runtime so
        # `import serving_runtime...` resolves in Model Serving.
        code_paths=[str(PACKAGE_ROOT)],
        signature=signature,
        input_example=input_example,
        pip_requirements=PIP_REQUIREMENTS,
    )
    run_id = run.info.run_id

print("Run ID:", run_id)
print("Logged model URI:", model_info.model_uri)


In [0]:
# ============================================================
# CELL 9 — LOCAL MLflow load test BEFORE Unity Catalog registration
# ============================================================
# This is the important V2 safety check: force MLflow to load the
# just-logged artifact and execute WorldBankGEPAgentModel.load_context().

loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
print("PASS: MLflow artifact loaded successfully")
print("The packaged serving_runtime import chain resolved correctly.")


In [0]:
# ============================================================
# CELL 10 — Register tested artifact in Unity Catalog
# ============================================================
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=MODEL_NAME,
)

MODEL_VERSION = str(registered_model.version)

print("\n==========================================")
print("MODEL REGISTERED")
print("==========================================")
print("Model:", MODEL_NAME)
print("Version:", MODEL_VERSION)
print("Model URI:", model_info.model_uri)
print("\nSTOP HERE. Use the printed MODEL_VERSION in Notebook 12.")
